In [1]:
# Section 1: 安装依赖
# 用 sys.executable 确保装到当前 kernel 对应的 Python，而不是系统其他 Python
import sys
!{sys.executable} -m pip install -U transformers datasets trl peft accelerate


  Using cached transformers-5.8.0-py3-none-any.whl.metadata (33 kB)
  Using cached datasets-4.8.5-py3-none-any.whl.metadata (19 kB)
  Using cached trl-1.4.0-py3-none-any.whl.metadata (11 kB)
  Using cached peft-0.19.1-py3-none-any.whl.metadata (15 kB)
Using cached transformers-5.8.0-py3-none-any.whl (10.6 MB)
Using cached datasets-4.8.5-py3-none-any.whl (528 kB)
Using cached trl-1.4.0-py3-none-any.whl (751 kB)
Using cached peft-0.19.1-py3-none-any.whl (680 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 4.6.1
    Uninstalling datasets-4.6.1:
      Successfully uninstalled datasets-4.6.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.3.0
    Uninstalling transformers-5.3.0:╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [transformers]
      Successfully uninstalled transformers-5.3.038;5;237m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [transformers]
  Attempting uninstall: trl━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [transformers]
    Found ex

In [2]:
import sys
print(sys.executable)

/Library/Frameworks/Python.framework/Versions/3.13/bin/python3.13


In [3]:
# 确认 kernel 使用的 Python 路径（用于排查环境问题）
import sys
print('Kernel Python:', sys.executable)


Kernel Python: /Library/Frameworks/Python.framework/Versions/3.13/bin/python3.13


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


In [5]:
# Section: 创建保存目录
import os
from datetime import datetime

# Aligns with config.py: LORA_ADAPTER_DIR = 'cathey_lora_adapter'
BASE_OUTPUT_DIR   = "./cathey_lora_adapter"
CHECKPOINT_DIR    = os.path.join(BASE_OUTPUT_DIR, "checkpoints")
FINAL_ADAPTER_DIR = os.path.join(BASE_OUTPUT_DIR, "final_adapter")

os.makedirs(BASE_OUTPUT_DIR,   exist_ok=True)
os.makedirs(CHECKPOINT_DIR,    exist_ok=True)
os.makedirs(FINAL_ADAPTER_DIR, exist_ok=True)

print("BASE_OUTPUT_DIR  =", BASE_OUTPUT_DIR)
print("CHECKPOINT_DIR   =", CHECKPOINT_DIR)
print("FINAL_ADAPTER_DIR=", FINAL_ADAPTER_DIR)


BASE_OUTPUT_DIR  = ./cathey_lora_adapter
CHECKPOINT_DIR   = ./cathey_lora_adapter/checkpoints
FINAL_ADAPTER_DIR= ./cathey_lora_adapter/final_adapter


In [6]:
# Section 2: 导入依赖

import os
import json
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

In [7]:
# Section 3: 基础配置

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

TRAIN_FILE        = "train.jsonl"
VAL_FILE          = "val.jsonl"
OUTPUT_DIR        = "./cathey_lora_adapter"         # matches config.py LORA_ADAPTER_DIR
FINAL_ADAPTER_DIR = OUTPUT_DIR + "/final_adapter"  # matches config.py LORA_INFERENCE_PATH
CHECKPOINT_DIR    = OUTPUT_DIR + "/checkpoints"

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

print("MODEL_NAME =", MODEL_NAME)
print("TRAIN_FILE =", TRAIN_FILE)
print("VAL_FILE   =", VAL_FILE)
print("OUTPUT_DIR =", OUTPUT_DIR)
print("DEVICE     =", DEVICE)


MODEL_NAME = Qwen/Qwen2.5-3B-Instruct
TRAIN_FILE = train.jsonl
VAL_FILE   = val.jsonl
OUTPUT_DIR = ./cathey_lora_adapter
DEVICE     = mps


In [8]:
# Section 3.5: 从 finetune/train_data.py 生成 train.jsonl / val.jsonl
# 已存在则跳过，强制重新生成请删掉两个 .jsonl 文件再跑此 cell
import sys, json, random as _rng

if os.path.exists(TRAIN_FILE) and os.path.exists(VAL_FILE):
    print(f"Files already exist — skipping generation.")
    print(f"  {TRAIN_FILE} ({sum(1 for _ in open(TRAIN_FILE))} lines)")
    print(f"  {VAL_FILE}   ({sum(1 for _ in open(VAL_FILE))} lines)")
else:
    sys.path.insert(0, '.')
    from finetune.train_data import RAW_TRAIN_DATA

    data = [{"input": inp, "output": out} for inp, out in RAW_TRAIN_DATA]
    _rng.seed(42)
    _rng.shuffle(data)

    n_val  = max(200, int(len(data) * 0.1))  # 10% val, at least 200
    val_d  = data[:n_val]
    train_d = data[n_val:]

    with open(TRAIN_FILE, 'w') as fh:
        for item in train_d:
            fh.write(json.dumps(item, ensure_ascii=False) + '\n')
    with open(VAL_FILE, 'w') as fh:
        for item in val_d:
            fh.write(json.dumps(item, ensure_ascii=False) + '\n')

    print(f"Generated:")
    print(f"  {TRAIN_FILE} — {len(train_d)} examples")
    print(f"  {VAL_FILE}   — {len(val_d)}  examples")


Generated:
  train.jsonl — 20250 examples
  val.jsonl   — 2250  examples


In [9]:
# Section 4: 检查训练文件

assert os.path.exists(TRAIN_FILE), f"Training file not found: {TRAIN_FILE}"
assert os.path.exists(VAL_FILE), f"Validation file not found: {VAL_FILE}"

print("Training and validation files found.")

Training and validation files found.


In [10]:
# Section 5: 加载 tokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded.")
print("Pad token:", tokenizer.pad_token)

Tokenizer loaded.
Pad token: <|endoftext|>


In [11]:
# Section 6: 加载数据集

dataset = load_dataset(
    "json",
    data_files={
        "train": TRAIN_FILE,
        "validation": VAL_FILE,
    }
)

print(dataset)
print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 20250
    })
    validation: Dataset({
        features: ['input', 'output'],
        num_rows: 2250
    })
})
Train size: 20250
Validation size: 2250


In [12]:
# Section 6.5: 补充/修正 general_qa 和 invalid 训练样本

import json

EXTRA_TRAIN_EXAMPLES = [
    # general_qa — greetings and personal questions (NOT invalid)
    {"input": "Cathey, hello!", "output": '{"type":"general_qa","answer":"Hello! How can I help you?"}'},
    {"input": "Cathey, never mind.", "output": '{"type":"general_qa","answer":"No problem, just let me know if you need anything."}'},
    {"input": "Cathey, okay thanks.", "output": '{"type":"general_qa","answer":"You\'re welcome!"}'},
    # general_qa — knowledge questions
    {"input": "Cathey, how do I eat an apple?", "output": '{"type":"general_qa","answer":"Wash it first, then eat it."}'},
    {"input": "Cathey, what time is it?", "output": '{"type":"general_qa","answer":"I don\'t have access to a clock, but you can check your phone."}'},
    {"input": "Cathey, can I eat leftovers after 3 days?", "output": '{"type":"general_qa","answer":"Yes, most cooked food is safe for 3-4 days in the fridge."}'},
    {"input": "Cathey, what's the weather like?", "output": '{"type":"general_qa","answer":"I don\'t have weather data, but you can check a weather app."}'},
    {"input": "Cathey, how do I boil an egg?", "output": '{"type":"general_qa","answer":"Place the egg in boiling water for 6-12 minutes depending on your preference."}'},
    {"input": "Cathey, what is 2 plus 2?", "output": '{"type":"general_qa","answer":"4."}'},
    # needs_clarification — vague temperature request
    {"input": "Cathey, lower the temperature.", "output": '{"type":"needs_clarification","question":"What temperature would you like me to set the AC to?","options":["lower_ac_temperature","raise_ac_temperature"]}'},
    {"input": "Cathey, make it warmer.", "output": '{"type":"needs_clarification","question":"Would you like me to raise the AC temperature or close the window?","options":["raise_ac_temperature","close_window"]}'},
    # invalid — truly unintelligible only
    {"input": "Hmm.", "output": '{"type":"invalid"}'},
    {"input": "Just testing.", "output": '{"type":"invalid"}'},
]

EXTRA_VAL_EXAMPLES = [
    {"input": "Cathey, how long should I microwave rice?", "output": '{"type":"general_qa","answer":"About 2-3 minutes, stirring halfway through."}'},
    {"input": "Cathey, hello there!", "output": '{"type":"general_qa","answer":"Hello! How can I help?"}'},
    {"input": "Cathey, raise the temperature a little.", "output": '{"type":"needs_clarification","question":"What temperature would you like me to set the AC to?","options":["lower_ac_temperature","raise_ac_temperature"]}'},
    {"input": "Hmm hmm.", "output": '{"type":"invalid"}'},
]

with open(TRAIN_FILE, "a", encoding="utf-8") as f:
    for ex in EXTRA_TRAIN_EXAMPLES:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

with open(VAL_FILE, "a", encoding="utf-8") as f:
    for ex in EXTRA_VAL_EXAMPLES:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"Appended {len(EXTRA_TRAIN_EXAMPLES)} train examples, {len(EXTRA_VAL_EXAMPLES)} val examples")

Appended 13 train examples, 4 val examples


In [13]:
# Section 7: 查看原始样本

print("Sample train row:")
print(dataset["train"][0])

Sample train row:
{'input': 'Hey Cathey, the room is over-illuminated.', 'output': '{"type": "needs_clarification", "question": "Would you like me to dim the light or close the curtain?", "options": ["dim_light", "close_curtain"]}'}


In [14]:
SYSTEM_PROMPT = """
Return exactly one JSON object.

Allowed outputs:

{"type":"direct_command","device":"light|curtain|window|ac","action":"turn_on|turn_off|set_brightness|rgb_cycle|open|close|set_position|set_temperature","value":null_or_int}

{"type":"needs_clarification","question":"...","options":["...","..."]}

{"type":"general_qa","answer":"..."}

{"type":"invalid"}

Rules:
- If the user directly names a device and a clear action, always output direct_command.
- If the user makes an indirect or ambiguous home-control request, output needs_clarification.
- If the user asks a normal non-device question, output general_qa.
- If there is no meaningful request, output invalid.
- For the same kind of ambiguous request, keep the question and options consistent.
- Output JSON only.
- No extra text.
""".strip()

In [15]:
# Section 9: 格式化训练样本
# Use tokenizer.apply_chat_template() — works correctly for any model
# (Qwen2.5 uses <|im_start|>/<|im_end|>, TinyLlama used <|system|>/<|user|>/<|assistant|>)

def format_example(example):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": example["input"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,  # False: we include the assistant turn
    )
    return {"text": text}

dataset = dataset.map(format_example)
print("Formatting done. Sample:")
print(dataset["train"][0]["text"][:300], "...")


Map:   0%|          | 0/20250 [00:00<?, ? examples/s]

Map:   0%|          | 0/2250 [00:00<?, ? examples/s]

Formatting done. Sample:
<|im_start|>system
Return exactly one JSON object.

Allowed outputs:

{"type":"direct_command","device":"light|curtain|window|ac","action":"turn_on|turn_off|set_brightness|rgb_cycle|open|close|set_position|set_temperature","value":null_or_int}

{"type":"needs_clarification","question":"...","options ...


In [16]:
# Section 10: 检查格式化后的样本

print("Formatted sample:")
print(dataset["train"][0]["text"])

Formatted sample:
<|im_start|>system
Return exactly one JSON object.

Allowed outputs:

{"type":"direct_command","device":"light|curtain|window|ac","action":"turn_on|turn_off|set_brightness|rgb_cycle|open|close|set_position|set_temperature","value":null_or_int}

{"type":"needs_clarification","question":"...","options":["...","..."]}

{"type":"general_qa","answer":"..."}

{"type":"invalid"}

Rules:
- If the user directly names a device and a clear action, always output direct_command.
- If the user makes an indirect or ambiguous home-control request, output needs_clarification.
- If the user asks a normal non-device question, output general_qa.
- If there is no meaningful request, output invalid.
- For the same kind of ambiguous request, keep the question and options consistent.
- Output JSON only.
- No extra text.<|im_end|>
<|im_start|>user
Hey Cathey, the room is over-illuminated.<|im_end|>
<|im_start|>assistant
{"type": "needs_clarification", "question": "Would you like me to dim the

In [17]:
# Section 11: 加载 base model

if torch.cuda.is_available():
    dtype = torch.float16
    dmap  = "auto"
elif torch.backends.mps.is_available():
    dtype = torch.float16
    dmap  = {"":"mps"}   # explicit MPS placement for Qwen2.5
else:
    dtype = torch.float32
    dmap  = "cpu"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    device_map=dmap,
)
print(f"Base model loaded on {next(model.parameters()).device} [{dtype}]")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Base model loaded on mps:0 [torch.float16]


In [18]:
# Section 12: 配置 LoRA

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)

model.print_trainable_parameters()

trainable params: 1,843,200 || all params: 3,087,781,888 || trainable%: 0.0597


In [19]:
# Section 13: 配置训练参数

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=5,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    bf16=False,
    report_to="none",
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [20]:
# Section 14: 构建 Trainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
)

print("Trainer ready.")

Adding EOS to train dataset:   0%|          | 0/20250 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/20250 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/2250 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2250 [00:00<?, ? examples/s]

Trainer ready.


In [ ]:
# Section 15: 开始训练

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
# Section 15b: 保存 LoRA adapter 和 tokenizer
import os

os.makedirs(FINAL_ADAPTER_DIR, exist_ok=True)
trainer.model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print(f"Adapter saved to {FINAL_ADAPTER_DIR}")
print("config.py LORA_INFERENCE_PATH already points here — no extra step needed.")

expected = ["adapter_config.json", "adapter_model.safetensors"]
for fn in expected:
    path = os.path.join(FINAL_ADAPTER_DIR, fn)
    print(f"  {'✓' if os.path.exists(path) else '✗'} {fn}")


In [ ]:
# Section: checkpoint 保存配置

training_args = SFTConfig(
    output_dir=CHECKPOINT_DIR,   # checkpoint 保存目录
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",       # 每个 epoch 自动保存 checkpoint
    save_total_limit=2,          # 最多保留 2 个 checkpoint
    fp16=torch.cuda.is_available(),
    bf16=False,
    report_to="none",
)

In [ ]:
def extract_first_json_object(text: str):
    start = text.find("{")
    if start == -1:
        return None

    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]

    return None


def has_complete_json_object(text: str) -> bool:
    start = text.find("{")
    if start == -1:
        return False

    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return True

    return False
from transformers import StoppingCriteria, StoppingCriteriaList

class JsonStopOnComplete(StoppingCriteria):
    def __init__(self, tokenizer, prompt_length: int):
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores, **kwargs):
        generated_ids = input_ids[0][self.prompt_length:]
        text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
        return has_complete_json_object(text)

In [ ]:
def run_inference(test_input: str, max_new_tokens: int = 80):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": test_input},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_length = inputs["input_ids"].shape[1]

    stopping_criteria = StoppingCriteriaList([
        JsonStopOnComplete(tokenizer, prompt_length)
    ])

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            stopping_criteria=stopping_criteria,
            use_cache=True,
        )

    generated_ids = outputs[0][prompt_length:]
    raw_output = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    json_str = extract_first_json_object(raw_output)
    return {"raw_output": raw_output, "first_json": json_str}

In [ ]:
# Section 18: 测试样例 (inference after training)

test_inputs = [
    "Cathey, turn on the light.",
    "Cathey, turn off the light.",
    "Cathey, open the curtain.",
    "Cathey, close the window.",
    "Cathey, set the AC to 24 degrees.",
    "Cathey, I feel a little bit cold.",
    "Cathey, I feel cold.",
    "Cathey, it's a bit dark.",
    "Cathey, this room feels dark.",
    "Cathey, how do I make tea?",
    "Cathey, hello!",
    "Turn on the light.",             # no wake word → should be invalid
]

for x in test_inputs:
    print("=" * 70)
    print("INPUT :", x)
    print("OUTPUT:", run_inference(x))


In [ ]:
# Section: checkpoint 保存配置

training_args = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=5,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    bf16=False,
    report_to="none",
)

In [ ]:
# Load fine-tuned adapter for inference (run this instead of loading base model)
# from transformers import AutoTokenizer, AutoModelForCausalLM
# from peft import PeftModel
# import torch
#
# MODEL_NAME   = "Qwen/Qwen2.5-3B-Instruct"
# ADAPTER_PATH = "./cathey_lora_adapter/final_adapter"  # matches config.py LORA_INFERENCE_PATH
#
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token
#
# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     torch_dtype=torch.float16,
#     device_map="auto",
# )
# model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
# model.eval()
